# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanay1122/flyrank-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**My pick: Lane 2 — Refresh / Content Opportunity Scoring** (provisional; the lane guide lets me confirm or swap it until the end of Week 4).

The lane's question: *which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?* I picked it because every part of the frame is real from day one: there is a named decision-maker (the content reviewer with a fixed weekly capacity), a named action they can take per page (refresh / expand / protect / prune / monitor), a named output shape (a ranked queue with reason codes a human can inspect), and the starter pipeline already ran this exact task end to end — so I have honest starting numbers and a transparent baseline to beat. There is also a clear 7-week growth path: the starter's label is a proxy ("declining", a current-window bucket), and the warehouse daily facts let me upgrade it to a future-window question — features from the prior 90 days, outcome in the next 30 — which is the trustworthy version of this lane.

Why not Lane 4 (CTR opportunity scoring), which I also considered: the starter slice's position tiers are noisy right now — the raw `position_tier` even lumps the 1,205 `avg_position = 0` ("no data") rows into `top_3`, and real `top_3` pages have a median of only ~53 impressions in the window, where one click swings the rate. Lane 4 needs the warehouse's cleaner daily columns before tier comparisons mean anything; Lane 2 can start today on the starter data, and I can have lanes 2 and 4 share machinery later.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Load the starter dataset and confirm the arena this lane works in.
from pathlib import Path
import subprocess

import pandas as pd

REPO_URL = "https://github.com/tanay1122/flyrank-ml.git"
DATA_PATH = Path("data") / "raw" / "content_refresh_anonymized.csv"


def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / DATA_PATH).exists():
            return candidate
    # Cloud-runtime fallback: e.g. the Colab badge opens just the notebook, not the repo.
    for parent in [Path("/content"), Path.cwd()]:
        target = parent / "flyrank-ml"
        if not target.parent.exists():
            continue
        if not (target / DATA_PATH).exists():
            print(f"Starter CSV not found locally; cloning the public repo into {target} ...")
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=False, capture_output=True)
        if (target / DATA_PATH).exists():
            return target
    raise FileNotFoundError("Starter CSV not found - open/run this notebook from inside the flyrank-ml repo.")


REPO = find_repo_root()
df = pd.read_csv(REPO / DATA_PATH)

rows, cols = df.shape
print(f"Starter slice: {rows:,} rows x {cols} columns, {df['client_id'].nunique()} pseudonymized clients")
print(f"One row = one content item (page): {rows:,} rows, {df['content_id'].nunique():,} unique content_id")
print("Pipeline pool (impressions_90d > 0 and content_age_days >= 90):",
      int(((df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)).sum()))

# Support the why-not-Lane-4 point in section 1 with computed numbers, not memory:
top3 = df[df["position_tier"] == "top_3"]
real_top3 = top3[top3["avg_position"] > 0]
print(f"Why-not-Lane-4 check: the raw top_3 tier holds {len(top3):,} rows, and "
      f"{int((top3['avg_position'] == 0).sum()):,} of them have avg_position = 0 (= 'no data', not rank three)")
print(f"  Position-3-and-better pages with data: median {real_top3['impressions_90d'].median():.0f} impressions "
      f"in the window — one click swings any rate at that volume")

Starter slice: 30,000 rows x 44 columns, 32 pseudonymized clients
One row = one content item (page): 30,000 rows, 30,000 unique content_id
Pipeline pool (impressions_90d > 0 and content_age_days >= 90): 30000
Why-not-Lane-4 check: the raw top_3 tier holds 2,321 rows, and 1,205 of them have avg_position = 0 (= 'no data', not rank three)
  Position-3-and-better pages with data: median 53 impressions in the window — one click swings any rate at that volume


## 2. The question: decision, action, cost of a wrong call

**Search question (Week-1 version, provisional):**
*For a content reviewer with a fixed weekly slot (~20 pages), which visible pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring — so the scarce review slots land where the evidence of real opportunity or decay is strongest?*

- **Decision it improves:** how the weekly content-review queue gets filled — concretely, what reaches the top ~20 of it.
- **Who acts, and how:** a content reviewer works the queue top down and chooses one action per page (refresh, expand, protect, prune, monitor), guided by reason codes they can inspect. The output is decision-support — a human checks every page before anything happens.
- **Unit of analysis:** one pseudonymized content item (a page) — one row per page, trailing-90-day totals (starter). Later in the track: client x content x day facts, which is what makes an honest future-window label possible.
- **Output:** a ranked queue — score, one suggested action per page, reason codes, a confidence label. Not a yes/no verdict, and never an automatic publish action.
- **Cost of a wrong call (both sides quantified below):**
  - *Wasted slot:* reviewer hours go to a page whose "signal" was really consolidation, seasonality, or plain noise — cheap per error, but repeated weekly at scale it is how trust in the tool dies.
  - *Missed decay:* a page still holding real search traffic keeps losing visibility while nobody is pointed at it — the expensive failure. In this slice that pool is big: 9,961 pages have >= 500 impressions in the window AND trend "down", and together they held 50.7% of all logged impressions.
- **Why not just "train a model":** the first product is a transparent rule score the reviewer can interrogate; a model only earns its slot by beating that score on a capacity-matched metric (precision@K, K = the realistic review size), because the queue is consumed top-down, not as whole-accuracy.

The frame paragraph (framing skill template):

> For a content reviewer deciding which ~20 pages per week to check first, we will build a ranked review queue from anonymized page-level search and engagement signals, scoring each page for opportunity/decay risk, measured by precision@K and average precision against a transparent rule-based baseline score. A wrong call costs reviewer hours on noise (false top) or a missed decay on a high-traffic page (false bottom). A plain rule is not enough because many weak, shifting signals — visibility, freshness, position, click-through, engagement — interact in heavy-tailed ways that hand-tuned weights cannot untangle. In Week 1 we claim only observed, directional, decision-support results.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Quantify the decision space the frame above is about.
imp = df["impressions_90d"]
trend = df["trend_direction"].astype(str).str.lower()

high_cost_pool = (trend == "down") & (imp >= 500)
share = 100 * imp[high_cost_pool].sum() / imp.sum()
print(f"High-cost pool (>=500 impressions AND trend 'down'): {int(high_cost_pool.sum()):,} pages")
print(f"They held {share:.1f}% of all logged impressions in the window")
print(f"Frame parameters: reviewer capacity ~20 pages/week -> K=20; metric candidates: precision@20, average precision")

High-cost pool (>=500 impressions AND trend 'down'): 9,961 pages
They held 50.7% of all logged impressions in the window
Frame parameters: reviewer capacity ~20 pages/week -> K=20; metric candidates: precision@20, average precision


## 3. Quick look at the data (2-3 real numbers)

Three numbers computed live from the starter CSV below (the model comparison in number 2 is read from the repo's verified committed output of the starter run, `outputs/model_report.md`):

1. **The rule flags are not selective.** 20,883 of 30,000 pages (69.6%) match at least one of the starter review rules; the flagged pages even hold 97.6% of the logged impressions. At ~20 reviews/week, clearing that list once takes ~20.1 years. So "filter by a rule" cannot be the deliverable — ranking under a tight capacity is the actual decision, and that is this lane's whole shape.
2. **A learned ranking beat the fixed score — once, on this slice.** The starter run's transparent rule baseline scored precision@50 = 0.24 (~12 of its top 50 actually positive); its random forest scored 0.74 (~37 of 50) on the same client-holdout split, using the starter's own label. That gap is the promise this lane is built on — but it was earned on a proxy label in a 30k-row slice, so my job over 7 weeks is to re-earn it honestly, not to copy it.
3. **The starter label is fragile — and that is the work.** 16,262 pages (54.2%) carry the "declining" label. With a base rate that high, naive accuracy looks great while saying little, and the label is a current-window bucket, not a future outcome. Upgrading the label first (features from prior 90 days -> outcome next 30 days, warehouse facts) is where most of the 7 weeks should go.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Starter review rules, mirrored EXACTLY from scripts/02_baseline_score.py, computed on the raw CSV.
import numpy as np

pos, age_days = df["avg_position"], df["content_age_days"]
word_count, ctr = df["word_count"], df["ctr"]
sessions, eng_rate, scroll = df["sessions_90d"], df["engagement_rate"], df["scroll_rate"]
days_since_update = df["days_since_last_update"]

rules = {
    "stale_visible_page":        (days_since_update >= 180) & (imp >= 500),
    "declining_with_demand":     (trend == "down") & (imp >= 100),
    "thin_visible_page":         (word_count > 0) & (word_count < 1200) & (imp >= 250),
    "page_one_decay_risk":       (pos > 0) & (pos <= 10) & (age_days >= 180),
    "low_ctr_visible_page":      (imp >= 500) & (pos > 0) & (pos <= 20) & (ctr < 0.5),
    "low_engagement_visible_page": (sessions >= 30) &
        (((eng_rate > 0) & (eng_rate < 30)) | ((scroll > 0) & (scroll < 30))),
}
any_flag = np.zeros(len(df), dtype=bool)
for name, mask in rules.items():
    mask = mask.fillna(False).to_numpy(dtype=bool)
    print(f"  {name:30s} {mask.sum():>7,} pages")
    any_flag |= mask

flagged = int(any_flag.sum())
print(f"  {'ANY starter rule':30s} {flagged:>7,} pages ({flagged/len(df)*100:.1f}% of the slice)")
print(f"  Flagged pages hold {imp[any_flag].sum() / imp.sum() * 100:.1f}% of the slice's logged impressions")
print(f"  At ~20 pages reviewed per week: {flagged / (20 * 52):.1f} years to clear this list once")

  stale_visible_page                  17 pages
  declining_with_demand           13,152 pages
  thin_visible_page                   82 pages
  page_one_decay_risk              7,076 pages
  low_ctr_visible_page             9,759 pages
  low_engagement_visible_page      6,508 pages
  ANY starter rule                20,883 pages (69.6% of the slice)
  Flagged pages hold 97.6% of the slice's logged impressions
  At ~20 pages reviewed per week: 20.1 years to clear this list once


In [4]:
# This cell is for CODE (numbers, a query, a check).
# The starter run's own numbers: its committed, verified report and the label's base rate.
import re
report = (REPO / "outputs" / "model_report.md").read_text(encoding="utf-8")

for line in report.splitlines():
    if re.match(r"^\|\s*(baseline_rules|logistic_regression|decision_tree|random_forest)\b", line):
        print(line.replace("|", " ").strip())
    elif re.search(r"(Rows scored|Declining-label rows|Declining-label rate|Split strategy)", line):
        print(line.replace("-", "", 1).strip())

base_rate = (trend == "down").mean()
declining = (trend == "down").sum()
print(f"Recomputed from the raw CSV: {int(declining):,} pages trend 'down' ({base_rate*100:.1f}%) — the base rate behind all of the above")

Rows scored: 30,000
Declining-label rows: 16,262
Declining-label rate: 0.542
Split strategy used for validation: client_holdout
decision_tree   0.742   0.575   0.540   0.716   0.634
logistic_regression   0.700   0.522   0.400   0.567   0.566
random_forest   0.750   0.618   0.740   0.744   0.640
baseline_rules   0.627   0.468   0.240   -   -
Recomputed from the raw CSV: 16,262 pages trend 'down' (54.2%) — the base rate behind all of the above


## 4. Careful words: what I can and can't claim

After honest validation (client-holdout split plus a leakage audit), this work will be able to say — scoped to this 30,000-page anonymized starter slice:

- "**We observed** pages with these signal patterns move together with the window's outcome" — measured associations, with effect sizes shown, not just a yes/no link.
- "**This ranking can be used to decide what to review first**" — decision-support, measured against a transparent baseline with precision@K, with the reason codes a reviewer can inspect.
- "**The learned ranking beat the fixed score on this slice**" — a comparison result, not a guarantee about any individual page.

It will never be able to say:

- that a refresh **caused** a recovery, or that acting on the queue caused anything — no experiment was (or can be) run on this data; the language of the final paper stays at "observed / directional / decision-support" unless a real causal design is added;
- anything about **"Google's algorithm" or AI citations/rankings** — none of these tables speak for a search engine, and AI-session counts measure click-throughs only;
- that the starter numbers **transfer** to the ~79M-row warehouse — they are one slice, and every result there has to be re-earned with proper validation;
- that "**declining" means future decline** — this slice's label is a current-window bucket with a 54.2% base rate, and it is exactly the weak proxy the next weeks should improve, not trust.

What would make the claims wrong — the leakage risks I will keep auditing every week: a feature computed over the outcome window; the label's own sources (`trend_direction`, `trend_pct`) sneaking into features; pages from the same client landing in both train and test without a grouped split; thresholds tuned against the test set. The cell below runs the first guard now.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Run a real leakage guard: the label's own sources must never appear among the model's features.
import sys

sys.path.insert(0, str(REPO / "scripts"))
from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES

label_sources = {"trend_direction", "trend_pct", "is_declining_label"}
model_features = set(MODEL_NUMERIC_FEATURES) | set(MODEL_CATEGORICAL_FEATURES)
overlap = sorted(label_sources & model_features)
assert not overlap, f"Label sources leaked into model features: {overlap}"
print(f"Guard passed: none of {sorted(label_sources)} appear among the {len(model_features)} starter model features")

Guard passed: none of ['is_declining_label', 'trend_direction', 'trend_pct'] appear among the 26 starter model features


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.